In [1]:
import pandas as pd
import numpy as np

In [6]:
df=pd.read_csv('vehicles.csv')
df['price_numeric']=pd.to_numeric(df['price'],errors='coerce')
df['year']=pd.to_numeric(df['year'],errors='coerce')
df.shape

/var/folders/jb/9m9w0dm52ps6xxz631ss34tm0000gn/T/ipykernel_8797/1075744981.py:1: DtypeWarning: Columns (4,32,40) have mixed types. Specify dtype option on import or set low_memory=False.
  df=pd.read_csv('vehicles.csv')


(295451, 42)

In [7]:
df.duplicated(subset=['year','manufacturer','model','odometer','price','state']).sum()

np.int64(84843)

In [8]:
df = df.drop_duplicates(subset=['year', 'manufacturer', 'model', 'odometer', 'price', 'state'])
df.shape

(210608, 42)

### Looking for Vehicles ID

In [11]:
df['VIN'].value_counts().head(10)

VIN
2C4RC3BG1TR171554    83
1FTWX33F9YEA12508    78
1D7KS28C26J203560    77
3D7KU28C84G150758    77
1GYS4GKL2RR104604    76
1FTHX26F6TEA44954    75
W1KVK8BBXPF015647    69
WA19FAFL1DA109333    69
1FTNW21FXYED30441    69
1GKS2DKT2NR136641    69
Name: count, dtype: int64

In [12]:
has_vin = df['VIN'].notnull()
df = pd.concat([df[has_vin].drop_duplicates(subset=['VIN']), df[~has_vin]])
df.shape

(187906, 42)

In [13]:
df['price_numeric'].isnull().sum()

np.int64(0)

In [14]:
(df['price_numeric'] == 0).sum()

np.int64(6181)

In [15]:
df = df[df['price_numeric'] > 0]
df.shape

(181725, 42)

### Looking for fake Odometer values 

In [16]:
df['odometer'].isnull().sum()

np.int64(1822)

In [23]:
fake_values = [999999, 1000000, 1111111, 1234567, 9999999, 10000000]
df.loc[df['odometer'].isin(fake_values), 'odometer'] = np.nan

In [24]:
df.loc[df['odometer'] > 1000000, 'odometer'] = np.nan

In [25]:
df.loc[(df['odometer'] < 100) & (df['year'] < 2025), 'odometer'] = np.nan

In [20]:
df.loc[(df['odometer_broken'] == 1) | (df['odometer_rolled_over'] == 1), 'odometer'] = np.nan

In [22]:
df['odometer'].isnull().mean()*100

np.float64(3.5971935617003714)

- fake values and placeholders are set as naN , not dropped .
-  3.5% now have missing mileage, to be imputed after train test split 

# Cylinders 

In [30]:
df['cylinders'].value_counts()

cylinders
4.0     50978
6.0     46168
8.0     33169
3.0      1060
5.0       993
10.0      727
12.0      192
Name: count, dtype: int64

In [32]:
# df['cylinders'] = df['cylinders'].str.split().str[0]
# df['cylinders'] = pd.to_numeric(df['cylinders'], errors='coerce')
# df['cylinders'].value_counts()

- only the number of cylinders are taken by converting into numeric 

# Base model 

In [33]:
words = df['model'].str.lower().str.strip().str.replace('-', '').str.split()
df['base_model'] = words.str[0]
df.loc[df['base_model'] == 'grand', 'base_model'] = words.str[0] + ' ' + words.str[1]
df['base_model'].value_counts().head(20)

base_model
f150              5694
silverado         5585
1500              3297
wrangler          3245
f250              2593
camry             2477
sierra            2465
accord            2372
civic             2275
tacoma            2151
escape            2063
explorer          2049
f350              1999
grand cherokee    1981
crv               1872
transit           1861
forester          1830
corolla           1793
outback           1760
2500              1692
Name: count, dtype: int64

- created base model so that f-150 and f150 be the same .

# Dropping not important columns 

In [34]:
df.shape

(181725, 43)

In [35]:
drop_cols = ['id', 'url', 'region_url', 'image_url', 'image_urls', 'VIN', 'description', 'title',
             'size', 'county', 'address_street', 'address_city', 'address_state', 'address_postal_code',
             'address_country', 'monthly_payment', 'other_attributes', 'price', 'model', 'region',
             'odometer_broken', 'odometer_rolled_over', 'updated_date']

df = df.drop(columns=drop_cols, errors='ignore')
df.shape

(181725, 20)

In [36]:
df.columns

Index(['year', 'manufacturer', 'condition', 'cylinders', 'fuel', 'odometer',
       'title_status', 'transmission', 'drive', 'type', 'paint_color', 'state',
       'lat', 'long', 'posting_date', 'seller_type', 'image_count',
       'delivery_available', 'price_numeric', 'base_model'],
      dtype='object')

- Dropped columns that can't help predict price: IDs, links, VIN, empty columns, raw text (description, title), and columns already replaced (price → price_numeric, model → base_model).

In [37]:
df.to_csv('cars_cleaned.csv', index=False)

In [38]:
df.isnull().mean() * 100

year                   0.003302
manufacturer           4.290549
condition             37.146788
cylinders             26.654560
fuel                   0.276792
odometer               3.597194
title_status           1.708626
transmission           0.330169
drive                 24.558536
type                   6.264961
paint_color           24.728023
state                  0.000000
lat                    0.036319
long                   0.036319
posting_date           0.000000
seller_type            0.000000
image_count            0.000000
delivery_available     0.000000
price_numeric          0.000000
base_model             1.258495
dtype: float64


- Removed 107,545 reposted duplicates and 6,181 zero-price ads: 295,451 → 181,725 listings.
- Set fake odometer readings to missing, converted cylinders to numbers, and created a cleaner base_model column.
- Dropped 23 unhelpful columns (43 → 20).
- Missing values range from 0% to 37% (condition). They will be filled inside a Pipeline after the train-test split: median for numbers, "unknown" for categories.
- Saved the result as new csv for future use.